# ***Importing Libraries***

In [2]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import sklearn
from sklearn.preprocessing import MinMaxScaler
from sklearn.metrics import mean_absolute_error, r2_score, mean_squared_error
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestRegressor
from xgboost import XGBRegressor

# ***Importing the dataset***

In [3]:
material_df=pd.read_csv("Ecopack-dataset.csv")
product_df=pd.read_csv("final_product_dataset.csv")

In [4]:
material_df.head(5)

,Material_Type,Tensile_Strength_MPa,Weight_Capacity_kg,Biodegradability_Score,CO2_Emission_Score,Recyclability_Percent,Moisture_Barrier_Grade,Cost_per_unit
0,Mushroom Mycelium,11.5,8.5,100.0,0.29,100.0,5,6.0580
1,Recycled Cardboard,16.5,7.7,94.8,0.88,95.4,4,7.4452
2,Bagasse (Sugarcane),22.4,0.7,96.5,1.20,63.2,5,7.2436
3,Virgin Plastic (PP),83.3,6.1,4.3,3.82,17.0,10,27.7500
4,Mushroom Mycelium,23.0,7.4,100.0,0.27,100.0,4,9.1740


In [5]:
product_df.head(5)

,product_category_name,product_weight_g,product_length_cm,product_height_cm,product_width_cm
0,clothing,960,24,47,12
1,clothing,3872,30,43,23
2,clothing,4526,33,7,26
3,clothing,869,39,42,6
4,clothing,1284,67,26,48


# ***Data Cleaning***

In [6]:
material_df.drop_duplicates(inplace=True)
product_df.drop_duplicates(inplace=True)

product_df.fillna(product_df.mean(numeric_only=True),inplace=True)
material_df.fillna(material_df.mean(numeric_only=True), inplace=True )

In [7]:
product_df.isnull().sum()

product_category_name    0
product_weight_g         0
product_length_cm        0
product_height_cm        0
product_width_cm         0
dtype: int64

In [8]:
material_df.isnull().sum()

Material_Type             0
Tensile_Strength_MPa      0
Weight_Capacity_kg        0
Biodegradability_Score    0
CO2_Emission_Score        0
Recyclability_Percent     0
Moisture_Barrier_Grade    0
Cost_per_unit             0
dtype: int64

## ***Feature Engineering***

In [9]:
#co2 Impact index

material_df['Co2_impact_index']=material_df['CO2_Emission_Score']*(1-material_df['Recyclability_Percent']/100)

In [10]:
material_df['Co2_impact_index']

0       0.00000
1       0.04048
2       0.44160
3       3.17060
4       0.00000
         ...   
4995    0.00000
4996    0.39072
4997    0.00000
4998    0.40530
4999    4.07862
Name: Co2_impact_index, Length: 5000, dtype: float64

In [11]:
#cost efficiency

material_df["Cost_Normalized"] = (
    material_df["Cost_per_unit"] - material_df["Cost_per_unit"].min()
) / (
    material_df["Cost_per_unit"].max() - material_df["Cost_per_unit"].min()
)

material_df["Cost_Efficiency_Index"] = 1 - material_df["Cost_Normalized"]

In [12]:
material_df["Cost_Efficiency_Index"]

0       0.862957
1       0.814704
2       0.821717
3       0.108416
4       0.754569
          ...   
4995    0.753595
4996    0.366738
4997    0.781422
4998    0.625366
4999    0.281245
Name: Cost_Efficiency_Index, Length: 5000, dtype: float64

# ***ML dataset preparation***

In [13]:
FEATURES = [
    "Tensile_Strength_MPa",
    "Weight_Capacity_kg",
    "Moisture_Barrier_Grade",
    "Biodegradability_Score",
    "Recyclability_Percent"
]

X = material_df[FEATURES]
y_cost = material_df["Cost_Efficiency_Index"]
y_co2 = material_df["Co2_impact_index"]

X_train, X_test, y_cost_train, y_cost_test = train_test_split(
    X, y_cost, test_size=0.2, random_state=42
)

_, _, y_co2_train, y_co2_test = train_test_split(
    X, y_co2, test_size=0.2, random_state=42
)

scaler = MinMaxScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

# ***Model Training***

In [14]:
# ---- Cost Model (Random Forest)
cost_model = RandomForestRegressor(
    n_estimators=200,
    max_depth=12,
    random_state=42
)
cost_model.fit(X_train_scaled, y_cost_train)

# ---- CO2 Model (XGBoost)
co2_model = XGBRegressor(
    n_estimators=250,
    learning_rate=0.08,
    max_depth=6,
    random_state=42
)
co2_model.fit(X_train_scaled, y_co2_train)


,"objective objective: typing.Union[str, xgboost.sklearn._SklObjWProto, typing.Callable[[typing.Any, typing.Any], typing.Tuple[numpy.ndarray, numpy.ndarray]], NoneType]Specify the learning task and the corresponding learning objective or a customobjective function to be used.For custom objective, see :doc:`/tutorials/custom_metric_obj` and:ref:`custom-obj-metric` for more information, along with the end note forfunction signatures.",'reg:squarederror'
,"base_score base_score: typing.Union[float, typing.List[float], NoneType]The initial prediction score of all instances, global bias.",None
,booster,None
,"callbacks callbacks: typing.Optional[typing.List[xgboost.callback.TrainingCallback]]List of callback functions that are applied at end of each iteration.It is possible to use predefined callbacks by using:ref:`Callback API `... note:: States in callback are not preserved during training, which means callback objects can not be reused for multiple training sessions without reinitialization or deepcopy... code-block:: python for params in parameters_grid: # be sure to (re)initialize the callbacks before each run callbacks = [xgb.callback.LearningRateScheduler(custom_rates)] reg = xgboost.XGBRegressor(**params, callbacks=callbacks) reg.fit(X, y)",None
,colsample_bylevel colsample_bylevel: typing.Optional[float]Subsample ratio of columns for each level.,None
,colsample_bynode colsample_bynode: typing.Optional[float]Subsample ratio of columns for each split.,None
,colsample_bytree colsample_bytree: typing.Optional[float]Subsample ratio of columns when constructing each tree.,None
,"device device: typing.Optional[str].. versionadded:: 2.0.0Device ordinal, available options are `cpu`, `cuda`, and `gpu`.",None
,"early_stopping_rounds early_stopping_rounds: typing.Optional[int].. versionadded:: 1.6.0- Activates early stopping. Validation metric needs to improve at least once in every **early_stopping_rounds** round(s) to continue training. Requires at least one item in **eval_set** in :py:meth:`fit`.- If early stopping occurs, the model will have two additional attributes: :py:attr:`best_score` and :py:attr:`best_iteration`. These are used by the :py:meth:`predict` and :py:meth:`apply` methods to determine the optimal number of trees during inference. If users want to access the full model (including trees built after early stopping), they can specify the `iteration_range` in these inference methods. In addition, other utilities like model plotting can also use the entire model.- If you prefer to discard the trees after `best_iteration`, consider using the callback function :py:class:`xgboost.callback.EarlyStopping`.- If there's more than one item in **eval_set**, the last entry will be used for early stopping. If there's more than one metric in **eval_metric**, the last metric will be used for early stopping.",None
,enable_categorical enable_categorical: boolSee the same parameter of :py:class:`DMatrix` for details.,False
,"eval_metric eval_metric: typing.Union[str, typing.List[typing.Union[str, typing.Callable]], typing.Callable, NoneType].. versionadded:: 1.6.0Metric used for monitoring the training result and early stopping. It can be astring or list of strings as names of predefined metric in XGBoost (See:doc:`/parameter`), one of the metrics in :py:mod:`sklearn.metrics`, or anyother user defined metric that looks like `sklearn.metrics`.If custom objective is also provided, then custom metric should implement thecorresponding reverse link function.Unlike the `scoring` parameter commonly used in scikit-learn, when a callableobject is provided, it's assumed to be a cost function and by default XGBoostwill minimize the result during early stopping.For advanced usage on Early stopping like directly choosing to maximize insteadof minimize, see :py:obj:`xgboost.callback.EarlyStopping`.See :doc:`/tutorials/custom_metric_obj` and :ref:`custom-obj-metric` for moreinformation... code-block:: python from sklearn.datasets import load_diabetes

# ***Model Evaluation***

In [15]:
cost_pred=cost_model.predict(X_test_scaled)
co2_pred=co2_model.predict(X_test_scaled)

In [16]:
print("Cost Model R2:", r2_score(y_cost_test, cost_pred))
print("Carbon Model R2:", r2_score(y_co2_test, co2_pred))

Cost Model R2: 0.9994824184709362
Carbon Model R2: 0.9557743454302261


# ***PRODUCT-AWARE RECOMMENDATION LOGIC***

In [17]:
product_df['product_weight_kg']=product_df['product_weight_g']/1000


In [18]:
# let us take first product
product=product_df.iloc[50]

#extracting the feasable materials

feasable_material=(material_df['Weight_Capacity_kg']>=product['product_weight_kg'])

#making prediction

feasable_material_df = material_df[feasable_material].copy()

x=feasable_material_df[FEATURES]
x_scaled=scaler.transform(x)

#co2 prediction

feasable_material_df['Co2_impact_index_pred']=co2_model.predict(x_scaled)

#cost prediction

feasable_material_df['cost_efficiency_pred']= cost_model.predict(x_scaled)

# introducing a new feature for better recommendation of materials "capacity_utilization"

feasable_material_df['capacity_utilization']=product['product_weight_kg']/feasable_material_df['Weight_Capacity_kg']

# Normalising

feasable_material_df['co2_norm']=(
    (feasable_material_df['Co2_impact_index_pred']- feasable_material_df['Co2_impact_index_pred'].min())/
    (feasable_material_df['Co2_impact_index_pred'].max()-feasable_material_df['Co2_impact_index_pred'].min())
)

feasable_material_df['cost_norm']=(
    (feasable_material_df['cost_efficiency_pred']-feasable_material_df['cost_efficiency_pred'].min())/
    (feasable_material_df['cost_efficiency_pred'].max()-feasable_material_df['cost_efficiency_pred'].min())
)

feasable_material_df['uitl_norm']=(
    (feasable_material_df['capacity_utilization']-feasable_material_df['capacity_utilization'].min())/
    (feasable_material_df['capacity_utilization'].max()-feasable_material_df['capacity_utilization'].min())
)
#final suitability score
feasable_material_df['suitability_score'] = (
    (0.4 * (1 - feasable_material_df['co2_norm'])) +
    (0.4 * (1 - feasable_material_df['cost_norm']))+
    (0.2 * feasable_material_df['uitl_norm'])
)

# ***Final Ranking***

In [19]:
top_materials = (feasable_material_df.sort_values(
    by='suitability_score', ascending=False
).drop_duplicates(subset='Material_Type', keep='first').head(3)
)


In [20]:
def final_ranking(index):
   print("Your input product is :" ,product_df.iloc[index,0])
   print("Best pakacaging materials according to your product is:")

   return top_materials[['Material_Type','Co2_impact_index_pred','cost_efficiency_pred','suitability_score']]


In [21]:
final_ranking(50)

Your input product is : glassware
Best pakacaging materials according to your product is:


,Material_Type,Co2_impact_index_pred,cost_efficiency_pred,suitability_score
2613,Bioplastic (PLA),1.001273,0.276890,0.793925
1736,Bamboo Fiber,0.562124,0.424526,0.787667
1365,Aluminum Foil,0.152382,0.609526,0.743858


# ***Saving the models***

In [22]:
import joblib

joblib.dump(co2_model,"co2_model")


['co2_model']

In [23]:
joblib.dump(cost_model,"cost_model")

['cost_model']

In [24]:
X_train_scaled

array([[0.26705882, 0.61052632, 1.        , 0.        , 0.922     ],
       [0.20823529, 0.81052632, 0.5       , 1.        , 0.68      ],
       [0.58352941, 0.87368421, 0.875     , 0.836     , 0.439     ],
       ...,
       [0.94588235, 0.23157895, 1.        , 0.007     , 0.285     ],
       [0.81411765, 0.06315789, 1.        , 0.01      , 0.137     ],
       [0.27529412, 0.48421053, 0.125     , 0.936     , 0.843     ]])